# 🎙️ AI Interviewer — Multi-Agent System

### Agent Architecture
```
  User (Gradio UI)
        │
        ▼
┌─────────────────────┐
│  Interviewer Agent  │  ◄── Central Orchestrator
│  (State Manager)    │       Tracks context, history,
└────────┬────────────┘       scores, difficulty level
         │
    ┌────┴────┐
    ▼         ▼
┌────────┐ ┌──────────────────┐
│ Answer │ │    Question      │
│ Review │ │   Generator      │
│ Agent  │ │    Agent         │
└────────┘ └──────────────────┘
```
- **Answer Review Agent** — Scores (0–10), gives structured feedback, flags weak areas
- **Question Generator Agent** — Picks next question based on difficulty, topic, and history
- **Interviewer Agent** — Orchestrates both, tracks full session state, produces final summary

## 1. Imports & Environment

In [1]:
from dotenv import load_dotenv
from agents import Runner, Agent, trace
from elevenlabs import ElevenLabs
from elevenlabs.play import play
from pypdf import PdfReader
from pydantic import BaseModel, Field
from typing import Optional
import gradio as gr
import asyncio
import os

load_dotenv(override=True)

c:\Users\Sayak\Desktop\Projects\Student's Corner\python-backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## 2. Load Resume

In [2]:
user_resume_render = PdfReader("user/Sayak_Resume.pdf")
user_resume = ""
for page in user_resume_render.pages:
    text = page.extract_text()
    if text:
        user_resume += text
print(user_resume)

Sayak Mitra Majumder
♂¶ap-¶arker-altPune /envel⌢pesayakmitra16@gmail.com ♂phone-alt6370275513
/linkedin-inSayak Mitra Majumder /githubReversibleWizard
Summary
Computer Science student with project experience in backend development using Python, Flask, and SQL. Worked on AI-powered
applications and data-driven projects, with strong interest in Intelligent Automation and emerging AI technologies. Eager to
contribute to scalable solutions and grow as a developer.
Education
Kalinga Institute of Industrial T echnology
B.Tech in Computer Science and Communication Engineering
Sept 2022 – Present
◦ CGP A:8.55/10.0
Internship
Keploy API F ellowship
Backend Fellow – Open Source & API Testing
June 2025 – July 2025
◦ Completed a series of hands-on assignments focused on API testing, mocking, and traffic replay using Keploy.
◦ Designed and tested APIs using Postman and integrated Keploy into the request-response workflow.
◦ Gained practical experience and knowledge in software testing and CI/CD pra

## 3. Pydantic Models

In [3]:
# ── Answer Review Agent output ──────────────────────────────────────────────
class AnswerReview(BaseModel):
    question: str = Field(description="The question that was asked")
    user_answer: str = Field(description="The raw answer provided by the candidate")
    score: int = Field(description="Score from 0 to 10 rating the quality of the answer")
    strengths: str = Field(description="What the candidate did well in their answer")
    weaknesses: str = Field(description="Specific areas where the answer was lacking")
    user_answer_review: str = Field(description="Detailed, constructive feedback on the answer")
    topic_covered: str = Field(description="The main topic/skill this question was about, e.g. 'Python', 'System Design'")
    difficulty: str = Field(description="Difficulty of the question: 'easy', 'medium', or 'hard'")


# ── Question Generator Agent output ─────────────────────────────────────────
class NextQuestion(BaseModel):
    question: str = Field(description="The next interview question to ask")
    topic: str = Field(description="Topic/skill this question targets")
    difficulty: str = Field(description="Difficulty level: 'easy', 'medium', or 'hard'")
    reasoning: str = Field(description="Why this question was chosen given the interview history")


# ── Final Interview Summary ──────────────────────────────────────────────────
class InterviewSummary(BaseModel):
    overall_score: float = Field(description="Average score across all answers (0–10)")
    total_questions: int = Field(description="Total number of questions asked")
    strong_topics: str = Field(description="Topics the candidate answered well")
    weak_topics: str = Field(description="Topics the candidate struggled with")
    hiring_recommendation: str = Field(description="'Strong Yes', 'Yes', 'Maybe', or 'No'")
    detailed_summary: str = Field(description="Full narrative performance summary for the interviewer")


# ── Interview Session State ──────────────────────────────────────────────────
class InterviewContext(BaseModel):
    history: list[AnswerReview] = Field(default=[], description="All reviewed Q&A pairs")
    current_question: str = Field(default="Introduce yourself", description="The active question")
    current_topic: str = Field(default="Introduction", description="Topic of current question")
    current_difficulty: str = Field(default="easy", description="Current difficulty level")
    question_count: int = Field(default=0, description="How many questions have been asked")
    topics_covered: list[str] = Field(default=[], description="List of topics already covered")
    is_completed: bool = Field(default=False, description="Whether the session is over")

## 4. Interview Configuration

In [4]:
# ── Configure these before starting ─────────────────────────────────────────
name                    = "Sayak"
time                    = "20 min"
user_experience         = "Student"
work_experience         = "0 years"
current_confidence_level = "Low"
target_role             = "Backend Developer / AI Engineer"
max_questions           = 8  # derived from time; adjust as needed

# Topics to cover in order of priority (Interviewer Agent will use this)
priority_topics = [
    "Introduction",
    "Projects",
    "Python / Backend",
    "Algorithms & Data Structures",
    "Databases",
    "System Design",
    "AI / LLMs",
    "Behavioural",
]

## 5. Agent Definitions

In [5]:
MODEL = "gpt-5-mini"

# ── 5a. Answer Review Agent ──────────────────────────────────────────────────
answer_review_instructions = f"""
You are an expert technical interviewer evaluating a candidate's answer.

Candidate profile:
- Name: {name}
- Experience level: {user_experience} ({work_experience} of work experience)
- Confidence level: {current_confidence_level}
- Target role: {target_role}

Your job:
1. Read the question and the candidate's answer carefully.
2. Score the answer from 0 to 10:
   - 0-3: Incorrect or very incomplete
   - 4-5: Partially correct, missing key points
   - 6-7: Good, minor gaps
   - 8-9: Strong, well-structured
   - 10: Excellent, comprehensive
3. Identify concrete strengths and weaknesses.
4. Write actionable, constructive feedback (user_answer_review) — be specific, not generic.
5. Label the topic (e.g. 'Python', 'Flask', 'Machine Learning', 'System Design').
6. Label the difficulty: 'easy', 'medium', or 'hard'.

Tone: Professional but encouraging. Remember the candidate is {current_confidence_level} in confidence.
Keep feedback concise — 3 to 6 sentences max.
"""

answer_review_agent = Agent(
    name="AnswerReviewAgent",
    instructions=answer_review_instructions,
    output_type=AnswerReview,
    model=MODEL,
)


# ── 5b. Question Generator Agent ────────────────────────────────────────────
question_generator_instructions = f"""
You are a smart interview question generator.

Candidate profile:
- Name: {name}
- Experience: {user_experience}, {work_experience}
- Target role: {target_role}
- Interview duration: {time}

Resume data:
{user_resume}

Priority topics to cover (in order): {priority_topics}

You will receive:
- The history of questions asked so far (with scores)
- Topics already covered
- The last answer's score and difficulty
- How many questions remain

Your job — generate the SINGLE BEST next question:
- NEVER repeat a question already in the history
- Adapt difficulty based on recent score:
    * Score ≥ 7 → increase difficulty or move to a harder subtopic
    * Score 4-6 → stay at same difficulty, probe deeper on same topic
    * Score ≤ 3 → decrease difficulty or pivot to a fresh topic
- Prioritise topics from the priority list that haven't been covered yet
- If few questions remain, prefer behavioral or wrap-up questions
- Questions must be grounded in the candidate's actual resume (projects, skills, tech stack)
- Keep questions short and specific — one question only, no multi-part questions

Output the question, its topic, difficulty, and your reasoning.
"""

question_generator_agent = Agent(
    name="QuestionGeneratorAgent",
    instructions=question_generator_instructions,
    output_type=NextQuestion,
    model=MODEL,
)


# ── 5c. Summary Agent (used at end of session) ───────────────────────────────
summary_agent_instructions = f"""
You are generating a post-interview performance report.

Candidate: {name}
Role applied for: {target_role}
Experience: {user_experience}, {work_experience}

You will receive the full interview history — all questions, answers, scores, and feedback.

Your job:
1. Calculate the average score across all answers.
2. Identify the strongest topics (score ≥ 7) and weakest topics (score ≤ 4).
3. Give a hiring recommendation: 'Strong Yes', 'Yes', 'Maybe', or 'No'.
4. Write a detailed, balanced performance narrative covering:
   - Technical depth and accuracy
   - Communication clarity
   - Areas of growth
   - Overall suitability for {target_role}

Be honest, specific, and fair. Use evidence from the interview history.
"""

summary_agent = Agent(
    name="SummaryAgent",
    instructions=summary_agent_instructions,
    output_type=InterviewSummary,
    model=MODEL,
)

## 6. Interviewer Agent — Core Orchestrator

In [6]:
# ── Global session context ───────────────────────────────────────────────────
context = InterviewContext()


class InterviewerAgent:
    """
    Central orchestrator of the interview session.

    Responsibilities:
    - Initiates the interview with the first prompt
    - Routes candidate answers to AnswerReviewAgent
    - Requests next questions from QuestionGeneratorAgent
    - Tracks all session state (history, scores, topics, difficulty)
    - Decides when to conclude and triggers SummaryAgent
    """

    def __init__(self, context: InterviewContext, max_questions: int = 8):
        self.ctx = context
        self.max_questions = max_questions

    # ── Step 1: Review the candidate's answer ────────────────────────────────
    async def _review_answer(self, question: str, user_answer: str) -> AnswerReview:
        prompt = f"""
Question asked: {question}

Candidate answer: {user_answer}
"""
        with trace("AnswerReview"):
            result = await Runner.run(answer_review_agent, prompt)
        return result.final_output

    # ── Step 2: Generate the next question ───────────────────────────────────
    async def _generate_next_question(self, review: AnswerReview) -> NextQuestion:
        history_summary = "\n".join([
            f"  Q{i+1} [{h.difficulty}] [{h.topic_covered}] Score:{h.score}/10 — {h.question}"
            for i, h in enumerate(self.ctx.history)
        ])
        topics_covered = ", ".join(self.ctx.topics_covered) if self.ctx.topics_covered else "None yet"
        remaining = self.max_questions - self.ctx.question_count

        prompt = f"""
Interview history so far:
{history_summary if history_summary else '  (no questions asked yet)'}

Last answer summary:
  - Question: {review.question}
  - Score: {review.score}/10
  - Difficulty: {review.difficulty}
  - Topic: {review.topic_covered}
  - Weaknesses flagged: {review.weaknesses}

Topics already covered: {topics_covered}
Questions remaining: {remaining}

Generate the best next question.
"""
        with trace("QuestionGenerator"):
            result = await Runner.run(question_generator_agent, prompt)
        return result.final_output

    # ── Step 3: Build the formatted response sent back to the UI ─────────────
    def _format_response(self, review: AnswerReview, next_q: NextQuestion) -> str:
        score_bar = "█" * review.score + "░" * (10 - review.score)
        return (
            f"**📊 Score: {review.score}/10** `{score_bar}`\n\n"
            f"**✅ Strengths:** {review.strengths}\n\n"
            f"**⚠️ Improve:** {review.weaknesses}\n\n"
            f"**💬 Feedback:** {review.user_answer_review}\n\n"
            f"---\n"
            f"**🎯 Next [{next_q.difficulty.upper()}] — {next_q.topic}**\n\n"
            f"{next_q.question}"
        )

    # ── Step 4: Generate final interview summary ──────────────────────────────
    async def _generate_summary(self) -> str:
        history_text = "\n\n".join([
            f"Q{i+1}: {h.question}\n"
            f"Answer: {h.user_answer}\n"
            f"Score: {h.score}/10 | Topic: {h.topic_covered} | Difficulty: {h.difficulty}\n"
            f"Feedback: {h.user_answer_review}"
            for i, h in enumerate(self.ctx.history)
        ])

        with trace("InterviewSummary"):
            result = await Runner.run(summary_agent, history_text)

        s = result.final_output
        scores = [h.score for h in self.ctx.history]
        avg = sum(scores) / len(scores) if scores else 0

        score_bar = "█" * int(avg) + "░" * (10 - int(avg))
        return (
            f"## 🏁 Interview Complete!\n\n"
            f"**Overall Score: {avg:.1f}/10** `{score_bar}`\n\n"
            f"**Questions Answered:** {s.total_questions}\n\n"
            f"**✅ Strong Topics:** {s.strong_topics}\n\n"
            f"**⚠️ Weak Topics:** {s.weak_topics}\n\n"
            f"**🤝 Hiring Recommendation:** `{s.hiring_recommendation}`\n\n"
            f"---\n"
            f"### 📋 Full Report\n\n"
            f"{s.detailed_summary}"
        )

    # ── Main entry point called by the UI ────────────────────────────────────
    async def handle_response(self, user_message: str) -> str:
        """
        Called for every user message. Orchestrates:
          1. Review answer  →  2. Decide next step  →  3. Generate question or summary
        """
        # Session already finished
        if self.ctx.is_completed:
            return "The interview session has ended. Refresh the page to start a new session."

        question = self.ctx.current_question

        # ── 1. Review the answer ─────────────────────────────────────────────
        review = await self._review_answer(question, user_message)

        # ── 2. Update session state ──────────────────────────────────────────
        self.ctx.history.append(review)
        self.ctx.question_count += 1
        if review.topic_covered not in self.ctx.topics_covered:
            self.ctx.topics_covered.append(review.topic_covered)

        # ── 3. Decide: end session or continue? ──────────────────────────────
        if self.ctx.question_count >= self.max_questions:
            self.ctx.is_completed = True
            return await self._generate_summary()

        # ── 4. Generate the next question ────────────────────────────────────
        next_q = await self._generate_next_question(review)

        # ── 5. Update context for next round ─────────────────────────────────
        self.ctx.current_question  = next_q.question
        self.ctx.current_topic     = next_q.topic
        self.ctx.current_difficulty = next_q.difficulty

        # ── 6. Return formatted response to UI ───────────────────────────────
        return self._format_response(review, next_q)

In [7]:
# ── Global session context ───────────────────────────────────────────────────
context = InterviewContext()


class InterviewerAgent:
    """
    Central orchestrator of the interview session.

    Responsibilities:
    - Initiates the interview with the first prompt
    - Routes candidate answers to AnswerReviewAgent
    - Requests next questions from QuestionGeneratorAgent
    - Tracks all session state (history, scores, topics, difficulty)
    - Decides when to conclude and triggers SummaryAgent
    """

    def __init__(self, context: InterviewContext, max_questions: int = 8):
        self.ctx = context
        self.max_questions = max_questions

    # ── Step 1: Review the candidate's answer ────────────────────────────────
    async def _review_answer(self, question: str, user_answer: str) -> AnswerReview:
        prompt = f"""
Question asked: {question}

Candidate answer: {user_answer}
"""
        with trace("AnswerReview"):
            result = await Runner.run(answer_review_agent, prompt)
        return result.final_output

    # ── Step 2: Generate the next question ───────────────────────────────────
    async def _generate_next_question(self, review: AnswerReview) -> NextQuestion:
        history_summary = "\n".join([
            f"  Q{i+1} [{h.difficulty}] [{h.topic_covered}] Score:{h.score}/10 — {h.question}"
            for i, h in enumerate(self.ctx.history)
        ])
        topics_covered = ", ".join(self.ctx.topics_covered) if self.ctx.topics_covered else "None yet"
        remaining = self.max_questions - self.ctx.question_count

        prompt = f"""
Interview history so far:
{history_summary if history_summary else '  (no questions asked yet)'}

Last answer summary:
  - Question: {review.question}
  - Score: {review.score}/10
  - Difficulty: {review.difficulty}
  - Topic: {review.topic_covered}
  - Weaknesses flagged: {review.weaknesses}

Topics already covered: {topics_covered}
Questions remaining: {remaining}

Generate the best next question.
"""
        with trace("QuestionGenerator"):
            result = await Runner.run(question_generator_agent, prompt)
        return result.final_output

    # ── Step 3: Return ONLY the next question to the UI ─────────────────────
    # Reviews/scores are stored in context.history and shown only at the end
    def _format_next_question(self, next_q: NextQuestion) -> str:
        q_num = self.ctx.question_count + 1
        return (
            f"**Question {q_num} [{next_q.difficulty.upper()}] — {next_q.topic}**\n\n"
            f"{next_q.question}"
        )

    # ── Step 4: Generate final interview summary ──────────────────────────────
    async def _generate_summary(self) -> str:
        history_text = "\n\n".join([
            f"Q{i+1}: {h.question}\n"
            f"Answer: {h.user_answer}\n"
            f"Score: {h.score}/10 | Topic: {h.topic_covered} | Difficulty: {h.difficulty}\n"
            f"Feedback: {h.user_answer_review}"
            for i, h in enumerate(self.ctx.history)
        ])

        with trace("InterviewSummary"):
            result = await Runner.run(summary_agent, history_text)

        s = result.final_output
        scores = [h.score for h in self.ctx.history]
        avg = sum(scores) / len(scores) if scores else 0

        score_bar = "█" * int(avg) + "░" * (10 - int(avg))
        return (
            f"## 🏁 Interview Complete!\n\n"
            f"**Overall Score: {avg:.1f}/10** `{score_bar}`\n\n"
            f"**Questions Answered:** {s.total_questions}\n\n"
            f"**✅ Strong Topics:** {s.strong_topics}\n\n"
            f"**⚠️ Weak Topics:** {s.weak_topics}\n\n"
            f"**🤝 Hiring Recommendation:** `{s.hiring_recommendation}`\n\n"
            f"---\n"
            f"### 📋 Full Report\n\n"
            f"{s.detailed_summary}"
        )

    # ── Main entry point called by the UI ────────────────────────────────────
    async def handle_response(self, user_message: str) -> str:
        """
        Called for every user message. Orchestrates:
          1. Review answer  →  2. Decide next step  →  3. Generate question or summary
        """
        # Session already finished
        if self.ctx.is_completed:
            return "The interview session has ended. Refresh the page to start a new session."

        question = self.ctx.current_question

        # ── 1. Review the answer ─────────────────────────────────────────────
        review = await self._review_answer(question, user_message)

        # ── 2. Update session state ──────────────────────────────────────────
        self.ctx.history.append(review)
        self.ctx.question_count += 1
        if review.topic_covered not in self.ctx.topics_covered:
            self.ctx.topics_covered.append(review.topic_covered)

        # ── 3. Decide: end session or continue? ──────────────────────────────
        if self.ctx.question_count >= self.max_questions:
            self.ctx.is_completed = True
            return await self._generate_summary()

        # ── 4. Generate the next question ────────────────────────────────────
        next_q = await self._generate_next_question(review)

        # ── 5. Update context for next round ─────────────────────────────────
        self.ctx.current_question  = next_q.question
        self.ctx.current_topic     = next_q.topic
        self.ctx.current_difficulty = next_q.difficulty

        # ── 6. Return ONLY the next question to the UI ───────────────────────
        return self._format_next_question(next_q)

    # ── Manual end — triggered by 'End Interview' button ─────────────────────
    async def handle_end_interview(self) -> str:
        """Called when the user manually clicks End Interview."""
        if self.ctx.is_completed:
            return "The interview has already ended."
        if not self.ctx.history:
            return "⚠️ No answers recorded yet — answer at least one question before ending."
        self.ctx.is_completed = True
        return await self._generate_summary()

## 7. ElevenLabs TTS + STT — Voice & Text Interface

In [8]:
import re
import time
import queue
import threading
import numpy as np
import soundfile as sf
from elevenlabs.conversational_ai.conversation import Conversation, AudioInterface

# ─────────────────────────────────────────────────────────────────────────────
# ⚙️  CREDENTIALS — fill these in after creating your ElevenLabs agent
# ─────────────────────────────────────────────────────────────────────────────
# Add to your .env:
#   ELEVENLABS_API_KEY   = sk-...
#   ELEVENLABS_AGENT_ID  = <agent_id from ElevenLabs dashboard>

AGENT_ID  = os.getenv("ELEVENLABS_AGENT_ID")   # Voice Delivery Coach agent
VOICE_ID  = "onwK4e9ZLuTAKqWW03F9"             # TTS voice for interviewer replies
TTS_MODEL = "eleven_multilingual_v2"
STT_MODEL = "scribe_v1"                         # Scribe for clean transcription

FILLER_WORDS = {"um","uh","like","you know","so","basically",
                "actually","literally","right","okay","hmm"}


# ─────────────────────────────────────────────────────────────────────────────
# ElevenLabs client
# ─────────────────────────────────────────────────────────────────────────────
eLabs = ElevenLabs(api_key=os.getenv("ELEVENLABS_API_KEY"))


# ─────────────────────────────────────────────────────────────────────────────
# TTS helper
# ─────────────────────────────────────────────────────────────────────────────
def _strip_markdown(text: str) -> str:
    text = re.sub(r"#{1,6}\s*", "", text)
    text = re.sub(r"\*{1,2}([^*]+)\*{1,2}", r"\1", text)
    text = re.sub(r"`[^`]*`", "", text)
    text = re.sub(r"---+", "", text)
    text = re.sub(r"\n{2,}", ". ", text)
    return text.strip()

def speak(text: str) -> None:
    """Convert AI text → speech via ElevenLabs (non-blocking)."""
    def _run():
        try:
            audio = eLabs.text_to_speech.convert(
                text=_strip_markdown(text),
                voice_id=VOICE_ID,
                model_id=TTS_MODEL,
            )
            play(audio)
        except Exception as e:
            print(f"[ElevenLabs TTS] {e}")
    threading.Thread(target=_run, daemon=True).start()


# ─────────────────────────────────────────────────────────────────────────────
# Scribe STT — clean transcript only (no emotion, just text)
# ─────────────────────────────────────────────────────────────────────────────
def transcribe(audio_path: str) -> str:
    """Get clean transcript via ElevenLabs Scribe."""
    with open(audio_path, "rb") as f:
        result = eLabs.speech_to_text.convert(file=f, model_id=STT_MODEL)
    return result.text.strip()


# ─────────────────────────────────────────────────────────────────────────────
# FileAudioInterface — streams a recorded file into the ElevenLabs agent
# ─────────────────────────────────────────────────────────────────────────────
class FileAudioInterface(AudioInterface):
    """
    Replaces the real-time microphone interface with a pre-recorded file.
    Streams the file's PCM bytes to the agent, then captures the agent's
    spoken response as text via an internal queue.
    """

    TARGET_SR   = 16_000   # ElevenLabs agent expects 16 kHz PCM
    CHUNK_FRAMES = 1_600   # 100 ms chunks

    def __init__(self, audio_path: str):
        self.audio_path   = audio_path
        self._response_q  = queue.Queue()   # agent's spoken reply chunks
        self._response_text = ""
        self._stop_event  = threading.Event()

    # ── AudioInterface protocol ───────────────────────────────────────────────
    def start(self, input_callback):
        """Called by Conversation to start feeding audio in."""
        def _stream():
            audio, sr = sf.read(self.audio_path, dtype="int16", always_2d=False)
            # Resample to 16 kHz mono if needed
            if sr != self.TARGET_SR:
                from scipy.signal import resample
                audio = resample(audio, int(len(audio) * self.TARGET_SR / sr)).astype(np.int16)
            if audio.ndim > 1:
                audio = audio[:, 0]

            for i in range(0, len(audio), self.CHUNK_FRAMES):
                if self._stop_event.is_set():
                    break
                chunk = audio[i : i + self.CHUNK_FRAMES]
                input_callback(chunk.tobytes())
                time.sleep(self.CHUNK_FRAMES / self.TARGET_SR)

            # Small silence buffer so the agent has time to finalise
            silence = np.zeros(self.TARGET_SR, dtype=np.int16)
            input_callback(silence.tobytes())

        threading.Thread(target=_stream, daemon=True).start()

    def stop(self):
        self._stop_event.set()

    def output(self, audio: bytes):
        """Agent's audio reply — we discard the audio but later read the text."""
        pass

    def interrupt(self):
        pass


# ─────────────────────────────────────────────────────────────────────────────
# ElevenLabs Agent voice analysis
# ─────────────────────────────────────────────────────────────────────────────
def analyze_voice_with_agent(audio_path: str) -> str:
    """
    Streams the recorded audio to the ElevenLabs Voice Delivery Coach agent.
    The agent hears the actual voice (tone, energy, pace, hesitation) and
    returns a structured delivery review as plain text.

    Returns the agent's analysis text, or an error string on failure.
    """
    if not AGENT_ID:
        return "⚠️ ELEVENLABS_AGENT_ID not set — add it to your .env file."

    collected_text: list[str] = []
    done_event = threading.Event()

    def on_agent_response(response):
        collected_text.append(response)

    def on_conversation_end():
        done_event.set()

    audio_iface = FileAudioInterface(audio_path)

    try:
        conv = Conversation(
            client           = eLabs,
            agent_id         = AGENT_ID,
            audio_interface  = audio_iface,
            callback_agent_response          = on_agent_response,
            callback_conversation_ended      = on_conversation_end,
        )
        conv.start_session()
        # Wait for the agent to finish — max 30 s
        done_event.wait(timeout=30)
        conv.end_session()
    except Exception as e:
        return f"[ElevenLabs Agent error] {e}"

    return " ".join(collected_text).strip() or "[No response from agent]"


def _format_agent_review(agent_text: str) -> str:
    """Wrap the agent's plain-text analysis in a tidy Markdown block."""
    return (
        "### 🎙️ Voice Delivery Review\n\n"
        "> *Analysed by ElevenLabs Voice Delivery Coach — "
        "heard your actual tone, energy & emotion*\n\n"
        f"{agent_text}"
    )


# ─────────────────────────────────────────────────────────────────────────────
# Session
# ─────────────────────────────────────────────────────────────────────────────
context     = InterviewContext()
interviewer = InterviewerAgent(context=context, max_questions=max_questions)

opening_message = (
    f"👋 Welcome, **{name}**! I'll be your interviewer today.\n\n"
    f"Introduce yourself."
)


# ─────────────────────────────────────────────────────────────────────────────
# Core respond helper (shared by voice & text paths)
# ─────────────────────────────────────────────────────────────────────────────
def _respond(user_text: str, history: list) -> tuple[list, str]:
    if not user_text or not user_text.strip():
        return history, ""
    ai_reply = asyncio.run(interviewer.handle_response(user_text))
    history = history + [
        {"role": "user",      "content": user_text},
        {"role": "assistant", "content": ai_reply},
    ]
    speak(ai_reply)
    return history, ""


# ─────────────────────────────────────────────────────────────────────────────
# Voice path — Scribe transcript + ElevenLabs agent delivery review (parallel)
# ─────────────────────────────────────────────────────────────────────────────
def handle_audio(audio_path: str, history: list):
    """
    1. Scribe  → clean transcript for the Interviewer Agent
    2. ElevenLabs Agent  → emotional / delivery analysis (hears raw voice)
    Both run concurrently so there is no extra wait.
    """
    if audio_path is None:
        return history, "", None, ""

    # ── Transcribe (Scribe) ──────────────────────────────────────────────────
    try:
        transcript = transcribe(audio_path)
    except Exception as e:
        return history, f"[Transcription failed: {e}]", None, ""

    # ── Run interview response + voice agent analysis concurrently ────────────
    async def _run_interview():
        return await interviewer.handle_response(transcript)

    def _run_voice_agent():
        return analyze_voice_with_agent(audio_path)

    # Interview response (async) and voice review (sync/threaded) in parallel
    voice_result_holder: list[str] = []

    def voice_thread():
        voice_result_holder.append(_run_voice_agent())

    vt = threading.Thread(target=voice_thread, daemon=True)
    vt.start()

    ai_reply = asyncio.run(_run_interview())
    vt.join(timeout=35)   # wait for agent — generous timeout

    agent_review = voice_result_holder[0] if voice_result_holder else "[No voice review received]"

    # ── Update chat history ───────────────────────────────────────────────────
    history = history + [
        {"role": "user",      "content": transcript},
        {"role": "assistant", "content": ai_reply},
    ]
    speak(ai_reply)

    review_md = _format_agent_review(agent_review)
    return history, transcript, None, review_md


# ─────────────────────────────────────────────────────────────────────────────
# Text fallback path
# ─────────────────────────────────────────────────────────────────────────────
def handle_text(user_text: str, history: list) -> tuple[list, str]:
    return _respond(user_text, history)


# ─────────────────────────────────────────────────────────────────────────────
# Gradio Blocks UI
# ─────────────────────────────────────────────────────────────────────────────
with gr.Blocks(title="🎙️ AI Interviewer") as demo:
    gr.Markdown("# 🎙️ AI Interviewer")
    gr.Markdown(
        "Speak your answer **or** type it below. "
        "The interviewer will reply in text and voice — "
        "and your vocal delivery (tone, confidence, emotion) will be "
        "reviewed by an ElevenLabs agent after each spoken answer."
    )

    chatbot = gr.Chatbot(
        value=[{"role": "assistant", "content": opening_message}],
        height=400,
        label="Interview",
    )

    with gr.Row():
        audio_input = gr.Audio(
            sources=["microphone"],
            type="filepath",
            label="🎤 Record your answer (click mic → speak → click stop)",
            interactive=True,
        )

    with gr.Row():
        text_input = gr.Textbox(
            placeholder="Or type your answer here and press Enter / Send…",
            label="⌨️  Type your answer (fallback — no voice review)",
            lines=3,
            scale=8,
        )
        send_btn = gr.Button("Send ➤", variant="primary", scale=1)

    with gr.Accordion("🎙️ Voice Delivery Review  (ElevenLabs Agent)", open=False):
        voice_review_md = gr.Markdown(
            value="*Record a voice answer to see your delivery review here.*"
        )

    # ── Wire up events ────────────────────────────────────────────────────────
    audio_input.stop_recording(
        fn=handle_audio,
        inputs=[audio_input, chatbot],
        outputs=[chatbot, text_input, audio_input, voice_review_md],
    )

    text_input.submit(
        fn=handle_text,
        inputs=[text_input, chatbot],
        outputs=[chatbot, text_input],
    )
    send_btn.click(
        fn=handle_text,
        inputs=[text_input, chatbot],
        outputs=[chatbot, text_input],
    )

# Speak the opening prompt when the cell runs
speak(opening_message)

demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 8. Inspect Session History

In [9]:
# Run this after the interview to review all Q&A pairs and scores
for i, entry in enumerate(context.history):
    print(f"\n{'='*60}")
    print(f"Q{i+1} [{entry.difficulty.upper()}] | Topic: {entry.topic_covered} | Score: {entry.score}/10")
    print(f"Question : {entry.question}")
    print(f"Answer   : {entry.user_answer[:200]}..." if len(entry.user_answer) > 200 else f"Answer   : {entry.user_answer}")
    print(f"Strengths: {entry.strengths}")
    print(f"Weaknesses: {entry.weaknesses}")
    print(f"Feedback : {entry.user_answer_review}")


Q1 [EASY] | Topic: Self-Introduction | Score: 8/10
Question : Introduce yourself
Answer   : Hello, I'm Sayak Mitra Majumder, a final-year B.Tech student in Computer Science and Communication Engineering at KIIT University, with a CGPA of 8.55. \
I have experience in Python, Flask, and Django...
Strengths: Clear, concise structure; states education and CGPA upfront; lists relevant technical skills (Python, Flask, Django, AWS); mentions concrete projects and current interest in LLMs/Agentic AI.
Weaknesses: Lacks details about your specific role or concrete contributions in the projects; no metrics or outcomes to show impact; missing examples of tools/frameworks, internships, or links (e.g., GitHub) that would back up claims.
Feedback : Good concise introduction — you cover education, core skills, project experience, and a forward-looking interest area. To strengthen it further, add one sentence per project describing your specific contribution and any measurable outcome (e.g., accuracy 